# Get Physical Properties from AlphaFold Output

## Setup

In [1]:
%pip install -q matplotlib plotly Bio nglview

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================
# AlphaFold Binding Screener (CIF-based)
# Reads AlphaFold zip files directly
# Performs simple geometric binding analyses
# ============================================

# Cell 1: Imports & settings
import os
import zipfile
import pickle
import json
import shutil
import subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from Bio.PDB import MMCIFParser, PDBParser, ShrakeRupley, Superimposer, PDBIO
from Bio import pairwise2
from Bio.Seq import Seq
from Bio.SeqUtils import seq1

# Optional: MDAnalysis and NGLView (graceful fallback)
try:
    import MDAnalysis as mda
    MDANALYSIS_AVAILABLE = True
except Exception:
    MDANALYSIS_AVAILABLE = False

try:
    import nglview as nv
    NGLVIEW_AVAILABLE = True
except Exception:
    NGLVIEW_AVAILABLE = False

# ---------------- USER SETTINGS ----------------
zip_folder = Path("../Local/AlphaFoldRun/")    # folder with AlphaFold zip outputs
output_folder = Path("../Local/AlphaFoldMDA/af_binding_results")   # where outputs will be written
temp_extract_dir = Path("../Local/AlphaFoldMDA/af_extract_tmp")    # temporary extraction directory (kept until final cleanup)

contact_cutoff = 5.0   # Å for contact
clash_cutoff = 2.2     # Å for clash
score_weights = {
    "contacts": 0.4,
    "interface_area": 0.002,
    "clashes": -3.0,
    "centroid_distance": -0.2,
    "plddt_mean": 0.5   # small bonus for higher mean-confidence
}

# Optional reference structure for RMSD (PDB or CIF). Set to None to skip RMSD.
reference_path = None  # e.g. Path("reference_target.pdb")

# Create output / temp folders
output_folder.mkdir(parents=True, exist_ok=True)
temp_extract_dir.mkdir(parents=True, exist_ok=True)

# Parsers
parser_mmcif = MMCIFParser(QUIET=True)
parser_pdb = PDBParser(QUIET=True)

print("MDAnalysis available:", MDANALYSIS_AVAILABLE)
print("NGLView available:", NGLVIEW_AVAILABLE)


C:\Users\ryangustafson\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\Bio\pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


MDAnalysis available: True
NGLView available: True


In [3]:
# Cell 2: Helper functions

def load_structure_from_file(path):
    """Return a Biopython Structure parsed with MMCIF if .cif else PDBParser."""
    ext = path.suffix.lower()
    if ext == ".cif" or ext == ".mmcif":
        return parser_mmcif.get_structure(path.stem, str(path))
    else:
        return parser_pdb.get_structure(path.stem, str(path))


def write_structure_to_pdb(structure, out_path):
    """Write a Biopython structure to PDB with PDBIO."""
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(out_path))


def chain_centroid(chain):
    coords = np.array([atom.coord for atom in chain.get_atoms()])
    return coords.mean(axis=0) if coords.size else np.array([np.nan, np.nan, np.nan])


def compute_contacts_and_map(chainA, chainB, cutoff=5.0):
    """Return contacts list and residue distance matrix (min distance per residue pair)."""
    resA_list = [res for res in chainA if res.get_id()[0] == " "]
    resB_list = [res for res in chainB if res.get_id()[0] == " "]
    nA, nB = len(resA_list), len(resB_list)
    dist_map = np.full((nA, nB), np.inf)
    contacts = []
    for i, resA in enumerate(resA_list):
        atomsA = list(resA.get_atoms())
        for j, resB in enumerate(resB_list):
            atomsB = list(resB.get_atoms())
            min_d = np.inf
            for a in atomsA:
                for b in atomsB:
                    d = np.linalg.norm(a.coord - b.coord)
                    if d < min_d:
                        min_d = d
            dist_map[i, j] = min_d
            if min_d <= cutoff:
                contacts.append((resA.get_id()[1], resB.get_id()[1]))
    return contacts, resA_list, resB_list, dist_map


def detect_clashes(chainA, chainB, cutoff=2.2):
    clashes = 0
    for atomA in chainA.get_atoms():
        for atomB in chainB.get_atoms():
            if np.linalg.norm(atomA.coord - atomB.coord) < cutoff:
                clashes += 1
    return clashes


def compute_sasa_and_interface(structure, chainA_id="A", chainB_id="B"):
    """
    Compute per-atom SASA via ShrakeRupley on the whole structure.
    Return interface area (Å^2) between the two chains and per-residue SASA dicts.
    """
    try:
        sr = ShrakeRupley()
        sr.compute(structure, level="A")
    except Exception as e:
        print("ShrakeRupley failed:", e)
        return None, None, None

    model = next(structure.get_models())
    chainA = model[chainA_id]
    chainB = model[chainB_id]

    sasa_A = sum(getattr(atom, "sasa", 0.0) for atom in chainA.get_atoms())
    sasa_B = sum(getattr(atom, "sasa", 0.0) for atom in chainB.get_atoms())
    sasa_complex = sum(getattr(atom, "sasa", 0.0) for atom in model.get_atoms())
    interface_area = max((sasa_A + sasa_B - sasa_complex) / 2.0, 0.0)

    res_sasa_A = {res.get_id()[1]: sum(getattr(a, "sasa", 0.0) for a in res) for res in chainA if res.get_id()[0] == " "}
    res_sasa_B = {res.get_id()[1]: sum(getattr(a, "sasa", 0.0) for a in res) for res in chainB if res.get_id()[0] == " "}

    return interface_area, res_sasa_A, res_sasa_B


def extract_plddt_from_structure(structure):
    """Get per-residue mean pLDDT from B-factor column (best-effort)."""
    per_chain = {}
    model = next(structure.get_models())
    for chain in model:
        res_map = {}
        for res in chain:
            if res.get_id()[0] != " ":
                continue
            resnum = res.get_id()[1]
            vals = [atom.get_bfactor() for atom in res.get_atoms()]
            res_map[resnum] = float(np.mean(vals)) if vals else np.nan
        per_chain[chain.get_id()] = res_map
    return per_chain


def try_load_full_data_json(zip_path):
    """
    Look for *_full_data_0.json inside the zip and parse predicted_aligned_error (PAE) and pLDDT.
    For pLDDT mapping: we will attempt to split the flat pLDDT list across chains using chain
    sequence lengths (best-effort).
    """
    pae = None
    plddt_per_chain = None
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            cand = [n for n in z.namelist() if n.endswith("full_data_0.json")]
            if not cand:
                return None, None
            name = cand[0]
            with z.open(name) as fh:
                txt = fh.read().decode()
                j = json.loads(txt)
                # PAE
                if "pae" in j:
                    pae = np.array(j["pae"])
                # pLDDT: some AF versions store 'plddt' per residue
                if "plddt" in j:
                    plddt_flat = list(j["plddt"])
                    # Map to chains by splitting according to chain sequence lengths from the cif in the same zip
                    # Find cif
                    cif_candidates = [n for n in z.namelist() if n.endswith("_model_0.cif") or n.endswith("_model_0.cif.gz")]
                    if cif_candidates:
                        # parse sequences from cif via parser_mmcif (extract first and get chain sequences)
                        try:
                            # extract CIF to temp path to parse sequences
                            tmp_cif = temp_extract_dir / Path(cif_candidates[0]).name
                            with z.open(cif_candidates[0]) as src, open(tmp_cif, "wb") as out:
                                out.write(src.read())
                            struct = parser_mmcif.get_structure("tmp", str(tmp_cif))
                            model = next(struct.get_models())
                            # build lengths
                            lengths = [len([r for r in chain if r.get_id()[0] == " "]) for chain in model]
                            # split flat list into per-chain arrays
                            per_chain = {}
                            idx = 0
                            for chain, length in zip(model, lengths):
                                per_chain[chain.get_id()] = {}
                                slice_vals = plddt_flat[idx: idx + length]
                                # map to residue numbers in chain
                                res_list = [r.get_id()[1] for r in chain if r.get_id()[0] == " "]
                                for resnum, val in zip(res_list, slice_vals):
                                    per_chain[chain.get_id()][resnum] = float(val)
                                idx += length
                            plddt_per_chain = per_chain
                            # cleanup
                            try:
                                os.remove(tmp_cif)
                            except Exception:
                                pass
                        except Exception:
                            # fallback: leave pLDDT as None and let extract_plddt_from_structure handle bfactor
                            plddt_per_chain = None
    except Exception:
        return None, None
    return pae, plddt_per_chain


In [4]:
# Cell 3: Sequence alignment & RMSD helpers

def get_chain_sequence(chain):
    """Return one-letter sequence string for a Biopython chain (standard residues only)."""
    seq = []
    for res in chain:
        hetflag = res.get_id()[0]
        if hetflag != " ":
            continue
        resname = res.get_resname()
        try:
            seq.append(seq1(resname))
        except Exception:
            seq.append("X")
    return "".join(seq)


def align_sequences_and_map(ref_seq, mob_seq):
    """Global alignment (needleman-wunsch via pairwise2.align.globalxx) and return mapping of matched indices.
    Returns list of tuples (ref_index, mob_index) for aligned residues (1-based residue index positions in sequences).
    """
    # globalxx does match with score = number of matches. Use globalms if you need gap penalties.
    aln = pairwise2.align.globalxx(ref_seq, mob_seq, one_alignment_only=True)[0]
    ref_aln, mob_aln, score, start, end = aln
    mapping = []
    ref_i = 0
    mob_i = 0
    for a, b in zip(ref_aln, mob_aln):
        if a != "-" and b != "-":
            # indices are 0-based here; we'll return 0-based indices for sequence positions
            mapping.append((ref_i, mob_i))
        if a != "-":
            ref_i += 1
        if b != "-":
            mob_i += 1
    return mapping, aln


def compute_aligned_ca_rmsd(ref_chain, mob_chain, mapping):
    """Compute RMSD using CA atoms for residues matched by mapping (which uses 0-based sequence positions)."""
    ref_res = [r for r in ref_chain if r.get_id()[0] == " "]
    mob_res = [r for r in mob_chain if r.get_id()[0] == " "]
    ref_atoms = []
    mob_atoms = []
    for ref_idx, mob_idx in mapping:
        try:
            ref_ca = ref_res[ref_idx]["CA"]
            mob_ca = mob_res[mob_idx]["CA"]
            ref_atoms.append(ref_ca)
            mob_atoms.append(mob_ca)
        except Exception:
            continue
    if len(ref_atoms) < 3:
        return None  # not enough points for meaningful RMSD
    si = Superimposer()
    si.set_atoms(ref_atoms, mob_atoms)
    si.apply(mob_atoms)
    return float(si.rms)


def compute_rmsd_to_reference_by_alignment(ref_path, structure, chain_id="B"):
    """Load reference, align sequences to model chain_id, compute RMSD based on aligned CA atoms."""
    try:
        ref_struct = load_structure_from_file(Path(ref_path))
        ref_model = next(ref_struct.get_models())
        ref_chain = ref_model[chain_id]

        mob_model = next(structure.get_models())
        mob_chain = mob_model[chain_id]

        ref_seq = get_chain_sequence(ref_chain)
        mob_seq = get_chain_sequence(mob_chain)
        mapping, aln = align_sequences_and_map(ref_seq, mob_seq)
        if not mapping:
            return None
        rmsd = compute_aligned_ca_rmsd(ref_chain, mob_chain, mapping)
        return rmsd
    except Exception as e:
        print("RMSD calc failed:", e)
        return None


In [5]:
# Cell 3: RMSD functions (optional)
def compute_rmsd_between_residue_lists(ref_residues, mob_residues):
    """
    Compute RMSD after matching residues by sequence order.
    ref_residues, mob_residues: lists of Residue objects with same length and matching order.
    """
    ref_atoms = []
    mob_atoms = []
    for ref_res, mob_res in zip(ref_residues, mob_residues):
        # use CA atoms if present; otherwise use all heavy atoms in canonical order
        try:
            ref_ca = ref_res["CA"]
            mob_ca = mob_res["CA"]
            ref_atoms.append(ref_ca)
            mob_atoms.append(mob_ca)
        except KeyError:
            # fall back to all atoms (may create mismatch in indexing)
            ref_atoms.extend([a for a in ref_res.get_atoms()])
            mob_atoms.extend([a for a in mob_res.get_atoms()])

    R = Superimposer()
    ref_coords = np.array([atom.coord for atom in ref_atoms])
    mob_coords = np.array([atom.coord for atom in mob_atoms])
    R.set_atoms(ref_atoms, mob_atoms)
    R.apply(mob_atoms)
    return R.rms

def compute_rmsd_to_reference(ref_path, structure, chain_id="B"):
    """
    Compute backbone CA RMSD between reference chain (chain_id) loaded from ref_path and chain_id in structure.
    If reference has multiple chains or numbering mismatch, this will attempt to pair residues by sequence index.
    """
    try:
        ref_struct = load_structure_from_file(Path(ref_path))
        ref_model = next(ref_struct.get_models())
        ref_chain = ref_model[chain_id]
        mob_model = next(structure.get_models())
        mob_chain = mob_model[chain_id]

        # create lists of residues (only standard residues) and align by ordinal position
        ref_res = [r for r in ref_chain if r.get_id()[0] == " "]
        mob_res = [r for r in mob_chain if r.get_id()[0] == " "]
        n = min(len(ref_res), len(mob_res))
        if n == 0:
            return None
        rmsd = compute_rmsd_between_residue_lists(ref_res[:n], mob_res[:n])
        return rmsd
    except Exception as e:
        print("RMSD calculation failed:", e)
        return None


In [6]:
# Cell 4: FoldX & PRODIGY wrappers

def is_executable_in_path(name):
    """Return True if executable is in PATH."""
    from shutil import which
    return which(name) is not None

def run_foldx_analyse(pdb_path, out_dir):
    """
    Run FoldX RepairPDB and AnalyseComplex if foldx is available.
    Returns parsed results dict or None.
    """
    if not is_executable_in_path("foldx"):
        print("FoldX not found in PATH. Skipping FoldX step.")
        return None
    out_dir.mkdir(parents=True, exist_ok=True)
    pdb_name = Path(pdb_path).name
    cwd = out_dir
    # 1) RepairPDB
    try:
        subprocess.run(["foldx", "--command=RepairPDB", f"--pdb={pdb_name}"], cwd=str(out_dir), check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("FoldX RepairPDB failed:", e)
        return None
    # Find repaired file (FoldX naming convention: <pdb_name>_Repair.pdb)
    repaired = out_dir / f"{pdb_name.split('.pdb')[0]}_Repair.pdb"
    if not repaired.exists():
        # fallback: sometimes it's named differently; look for *_Repair.pdb
        repaired_candidates = list(out_dir.glob("*_Repair.pdb"))
        if repaired_candidates:
            repaired = repaired_candidates[0]
        else:
            print("FoldX Repair output not found.")
            return None
    # 2) AnalyseComplex
    try:
        subprocess.run(["foldx", "--command=AnalyseComplex", f"--pdb={repaired.name}"], cwd=str(out_dir), check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("FoldX AnalyseComplex failed:", e)
        return None
    # Parse FoldX output files (Interaction_XXXX or DifferencesBetweenComplexes.txt)
    # Look for "Interaction" files
    interaction_files = list(out_dir.glob("Interaction_*"))
    results = {}
    if interaction_files:
        # parse text lines for overall Interaction energy (FoldX file format varies)
        with open(interaction_files[0]) as fh:
            txt = fh.read()
        # Try to get a numeric value heuristically
        import re
        m = re.search(r"Total energy:\s*([-0-9]+\.[0-9]+)", txt)
        if m:
            results['foldx_interaction_energy'] = float(m.group(1))
        else:
            # attempt to find any float
            floats = re.findall(r"[-]?\d+\.\d+", txt)
            if floats:
                results['foldx_interaction_energy'] = float(floats[0])
    # cleanup intermediate files if needed (leave out_dir contents for inspection)
    return results

def run_prodigy_if_available(pdb_path, chainA="A", chainB="B"):
    """
    Attempt to run a local PRODIGY script or module if available.
    If not present, returns None. For many users, PRODIGY is easiest to run via its webserver.
    """
    # Try Python package import first
    try:
        import prodigy_notebook_api  # placeholder: many local installs differ
        # If you have a local module, call it appropriately here
        # This is just a placeholder; return None to indicate not run
        return None
    except Exception:
        pass
    # Try a prodigy executable
    if is_executable_in_path("prodigy"):
        try:
            res = subprocess.run(["prodigy", str(pdb_path), chainA, chainB], capture_output=True, check=True, text=True)
            out = res.stdout
            # parse out binding energy heuristically
            import re
            m = re.search(r"DeltaG.*?([-0-9]+\.[0-9]+)", out)
            if m:
                return {"prodigy_ddG": float(m.group(1))}
        except Exception as e:
            print("Calling PRODIGY failed:", e)
            return None
    else:
        print("PRODIGY not found as executable or Python module. Skipping PRODIGY step.")
        return None


In [7]:
# Cell 5: Main loop - process each ZIP and produce per-model outputs
zip_files = sorted(zip_folder.glob("*.zip"))
print(f"Found {len(zip_files)} zip files in {zip_folder}")

summary_rows = []
model_meta = {}
extracted_files_to_cleanup = []  # track extracted files and generated PDBs for final cleanup

for zip_path in zip_files:
    model_name = zip_path.stem
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            # find *_model_0.cif entry
            cif_candidates = [n for n in z.namelist() if n.endswith("_model_0.cif") or n.endswith("_model_0.cif.gz")]
            if not cif_candidates:
                print(f"  skipping {model_name}: no *_model_0.cif found")
                continue
            cif_entry = cif_candidates[0]
            extracted_cif_path = temp_extract_dir / Path(cif_entry).name
            # extract (overwrite if exists)
            with z.open(cif_entry) as source, open(extracted_cif_path, "wb") as out:
                out.write(source.read())
            extracted_files_to_cleanup.append(extracted_cif_path)

            # try load PAE and per-residue pLDDT from full_data_0.json if present
            pae, plddt_per_chain = try_load_full_data_json(zip_path)

        # parse CIF structure
        structure = parser_mmcif.get_structure(model_name, str(extracted_cif_path))
        model = next(structure.get_models())
        if 'A' not in [c.get_id() for c in model] or 'B' not in [c.get_id() for c in model]:
            print(f"  skipping {model_name}: missing chain A or B")
            continue

        chainA = model['A']
        chainB = model['B']

        # contacts/clashes/centroid
        contacts, resA_list, resB_list, dist_map = compute_contacts_and_map(chainA, chainB, contact_cutoff)
        n_contacts = len(contacts)
        unique_pairs = len(set(contacts))
        clashes = detect_clashes(chainA, chainB, clash_cutoff)
        cenA = chain_centroid(chainA)
        cenB = chain_centroid(chainB)
        centroid_distance = float(np.linalg.norm(cenA - cenB))

        # SASA
        interface_area, res_sasa_A, res_sasa_B = compute_sasa_and_interface(structure, "A", "B")
        if interface_area is None:
            interface_area = float("nan")

        # pLDDT: prefer JSON per-chain if available, otherwise read B-factor
        if plddt_per_chain:
            plddtA = plddt_per_chain.get('A', {})
            plddtB = plddt_per_chain.get('B', {})
        else:
            per_chain_plddt = extract_plddt_from_structure(structure)
            plddtA = per_chain_plddt.get('A', {})
            plddtB = per_chain_plddt.get('B', {})

        mean_plddt_A = float(np.nanmean(list(plddtA.values()))) if plddtA else float("nan")
        mean_plddt_B = float(np.nanmean(list(plddtB.values()))) if plddtB else float("nan")
        mean_plddt = float(np.nanmean([v for v in list(plddtA.values()) + list(plddtB.values()) if not np.isnan(v)])) if (plddtA or plddtB) else float("nan")

        # PAE present flag
        pae_present = False if pae is None else True

        # RMSD to reference via sequence alignment if requested
        rmsd_to_ref = None
        if reference_path is not None:
            rmsd_to_ref = compute_rmsd_to_reference_by_alignment(reference_path, structure, chain_id="B")

        # Sequence identities (for chain-level checks)
        seqA = get_chain_sequence(chainA)
        seqB = get_chain_sequence(chainB)

        # Optionally run MDAnalysis-based checks if available (e.g., hbonds)
        md_results = {}
        if MDANALYSIS_AVAILABLE:
            try:
                u = mda.Universe(str(extracted_cif_path))
                # you can add more MDAnalysis analyses here
                md_results['n_atoms'] = len(u.atoms)
            except Exception as e:
                md_results['error'] = str(e)

        # Totalscore (weights)
        ia_val = 0.0 if (np.isnan(interface_area)) else interface_area
        score = (n_contacts * score_weights['contacts']
                 + ia_val * score_weights['interface_area']
                 + clashes * score_weights['clashes']
                 + centroid_distance * score_weights['centroid_distance'])
        if not np.isnan(mean_plddt):
            score += mean_plddt * score_weights.get('plddt_mean', 0.0)

        # Save per-model PDB for downstream tools (FoldX/PRODIGY) if needed
        pdb_for_tools = temp_extract_dir / f"{model_name}.pdb"
        try:
            write_structure_to_pdb(structure, pdb_for_tools)
            extracted_files_to_cleanup.append(pdb_for_tools)
        except Exception as e:
            print("Could not write PDB for", model_name, ":", e)
            pdb_for_tools = None

        # Attempt FoldX / PRODIGY (only if executables present)
        foldx_res = None
        prodigy_res = None
        if pdb_for_tools and is_executable_in_path("foldx"):
            foldx_res = run_foldx_analyse(pdb_for_tools, temp_extract_dir / f"foldx_{model_name}")
        if pdb_for_tools:
            prodigy_res = run_prodigy_if_available(pdb_for_tools, chainA="A", chainB="B")

        # assemble meta & outputs
        row = {
            "Model": model_name,
            "Contacts": n_contacts,
            "Unique_Pairs": unique_pairs,
            "Interface_Area": float(interface_area) if not np.isnan(interface_area) else np.nan,
            "Centroid_Distance": centroid_distance,
            "Clashes": clashes,
            "Mean_pLDDT_A": mean_plddt_A,
            "Mean_pLDDT_B": mean_plddt_B,
            "Mean_pLDDT": mean_plddt,
            "RMSD_to_ref_B": rmsd_to_ref,
            "Score": score,
            "PAE_present": pae_present,
            "FoldX": foldx_res,
            "PRODIGY": prodigy_res
        }
        summary_rows.append(row)
        model_meta[model_name] = {
            "resA_list": [r.get_id()[1] for r in resA_list],
            "resB_list": [r.get_id()[1] for r in resB_list],
            "dist_map": dist_map,
            "plddtA": plddtA,
            "plddtB": plddtB,
            "res_sasa_A": res_sasa_A,
            "res_sasa_B": res_sasa_B,
            "pae": pae,
            "md_results": md_results
        }

        # Save per-model CSVs
        cmap_df = pd.DataFrame(dist_map,
                               index=[r.get_id()[1] for r in resA_list],
                               columns=[r.get_id()[1] for r in resB_list])
        cmap_df.to_csv(output_folder / f"{model_name}_contact_map.csv")

        int_res_rows = []
        for i, res in enumerate(resA_list):
            resnum = res.get_id()[1]
            min_dist = float(np.min(dist_map[i, :]))
            plddt = plddtA.get(resnum, np.nan)
            sasa = res_sasa_A.get(resnum, np.nan) if res_sasa_A else np.nan
            int_res_rows.append({"chain": "A", "resnum": resnum, "min_dist": min_dist, "plddt": plddt, "sasa": sasa})
        for j, res in enumerate(resB_list):
            resnum = res.get_id()[1]
            min_dist = float(np.min(dist_map[:, j]))
            plddt = plddtB.get(resnum, np.nan)
            sasa = res_sasa_B.get(resnum, np.nan) if res_sasa_B else np.nan
            int_res_rows.append({"chain": "B", "resnum": resnum, "min_dist": min_dist, "plddt": plddt, "sasa": sasa})
        int_df = pd.DataFrame(int_res_rows)
        int_df.to_csv(output_folder / f"{model_name}_per_residue_interface.csv", index=False)

        meta = {"model": model_name, "summary": row}
        with open(output_folder / f"{model_name}_meta.json", "w") as fh:
            json.dump(meta, fh, indent=2, default=lambda x: None)

        print(f"Processed {model_name}: score={score:.2f}, contacts={n_contacts}, interface={ia_val:.1f}, pLDDT(mean)={mean_plddt:.1f}")

    except Exception as e:
        print(f"Failed on {model_name}: {e}")

# Save summary CSV
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("Score", ascending=False)
summary_csv_path = output_folder / "binding_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nSummary saved to: {summary_csv_path}")

# Persist meta dict for future use
with open(output_folder / "model_meta.pkl", "wb") as fh:
    pickle.dump(model_meta, fh)

# Keep list of files to cleanup until final cleanup step
with open(output_folder / "extracted_files.txt", "w") as fh:
    for p in extracted_files_to_cleanup:
        fh.write(str(p) + "\n")

summary_df.head(40)


Found 252 zip files in ..\Local\AlphaFoldRun
PRODIGY not found as executable or Python module. Skipping PRODIGY step.
Processed fold_timp3_variant_adam10_ab_wt: score=78.94, contacts=98, interface=0.0, pLDDT(mean)=90.2
PRODIGY not found as executable or Python module. Skipping PRODIGY step.
Processed fold_timp3_variant_adam10_c_wt: score=78.94, contacts=98, interface=0.0, pLDDT(mean)=90.2
PRODIGY not found as executable or Python module. Skipping PRODIGY step.
Processed fold_timp3_variant_adam10_ef_wt: score=78.94, contacts=98, interface=0.0, pLDDT(mean)=90.2
PRODIGY not found as executable or Python module. Skipping PRODIGY step.
Processed fold_timp3_variant_adam17_ab_wt: score=82.45, contacts=109, interface=0.0, pLDDT(mean)=88.9
PRODIGY not found as executable or Python module. Skipping PRODIGY step.
Processed fold_timp3_variant_adam17_c_wt: score=82.45, contacts=109, interface=0.0, pLDDT(mean)=88.9
PRODIGY not found as executable or Python module. Skipping PRODIGY step.
Processed fo

,Model,Contacts,Unique_Pairs,Interface_Area,Centroid_Distance,Clashes,Mean_pLDDT_A,Mean_pLDDT_B,Mean_pLDDT,RMSD_to_ref_B,Score,PAE_present,FoldX,PRODIGY
93,fold_timp3_vs_adam17_c_aneglc_proteinmpnn_vadam17,114,114,0.000000e+00,26.480539,0,85.452923,92.152706,89.315861,None,84.961823,True,None,None
120,fold_timp3_vs_adam17_ef_tacd_proteinmpnn_vadam17,112,112,0.000000e+00,27.868567,0,85.544707,93.111111,89.907318,None,84.179946,True,None,None
125,fold_timp3_vs_adam17_ef_tscd_proteinmpnn_vmmp9,112,112,0.000000e+00,27.085501,0,85.519514,92.497897,89.543087,None,84.154443,True,None,None
19,fold_timp3_vs_adam10_ab_kgkyge_proteinmpnn_vad...,111,111,3.637979e-12,26.693525,0,87.107949,92.516514,90.124019,None,84.123305,True,None,None
117,fold_timp3_vs_adam17_ef_sncd_proteinmpnn_vmmp9,112,112,0.000000e+00,26.810165,0,85.075033,92.427391,89.314231,None,84.095082,True,None,None
119,fold_timp3_vs_adam17_ef_sscd_proteinmpnn_vmmp9,111,111,0.000000e+00,26.725742,0,85.820760,92.955145,89.934280,None,84.021991,True,None,None
124,fold_timp3_vs_adam17_ef_tscd_proteinmpnn_vadam17,111,111,0.000000e+00,27.320000,0,85.315584,92.398022,89.399152,None,83.635576,True,None,None
95,fold_timp3_vs_adam17_c_anenlc_proteinmpnn_vadam17,112,112,0.000000e+00,28.066290,0,84.565135,91.894011,88.790793,None,83.582139,True,None,None
112,fold_timp3_vs_adam17_ef_fscd_proteinmpnn_vadam17,110,110,0.000000e+00,27.318827,0,86.151245,92.917318,90.052404,None,83.562437,True,None,None
108,fold_timp3_vs_adam17_c_sveelc_proteinmpnn_vmmp9,112,112,0.000000e+00,27.392473,0,84.346878,91.200072,88.298269,None,83.470640,True,None,None


In [8]:
# Cell 6: Interactive Plotly exploration (Interface_Area vs Score)
if summary_df.shape[0] == 0:
    print("No results to plot.")
else:
    # Use reasonable columns, fill NaNs for plotting
    summary_df_plot = summary_df.copy()
    summary_df_plot["Interface_Area_filled"] = summary_df_plot["Interface_Area"].fillna(0.0)
    summary_df_plot["Mean_pLDDT_filled"] = summary_df_plot["Mean_pLDDT"].fillna(0.0)

    fig = px.scatter(summary_df_plot,
                     x="Interface_Area_filled",
                     y="Score",
                     size="Contacts",
                     color="Mean_pLDDT_filled",
                     hover_data=["Model", "Contacts", "Clashes", "Centroid_Distance", "RMSD_to_ref_B", "PAE_present"],
                     labels={"Interface_Area_filled": "Interface Area (Å²)", "Mean_pLDDT_filled": "Mean pLDDT"})
    fig.update_layout(title="AlphaFold binding screener: Interface Area vs Score",
                      legend_title_text="Mean pLDDT")

    html_out = output_folder / "binding_dashboard.html"
    fig.write_html(str(html_out))
    print(f"Interactive dashboard saved: {html_out}")
    fig.show()


Interactive dashboard saved: ..\Local\AlphaFoldMDA\af_binding_results\binding_dashboard.html


In [9]:
# Cell 7: NGLView — show the top-scoring model inside the notebook (optional)
if not NGLVIEW_AVAILABLE:
    print("nglview not available in this environment. Install it for inline visualization.")
else:
    if summary_df.shape[0] == 0:
        print("No models available to view.")
    else:
        top_model = summary_df.iloc[0]["Model"]
        print("Top model:", top_model)
        zip_candidates = list(zip_folder.glob(f"{top_model}*.zip"))
        if zip_candidates:
            zip_path = zip_candidates[0]
            with zipfile.ZipFile(zip_path, "r") as z:
                candidates = [n for n in z.namelist() if n.endswith("_model_0.cif")]
                if candidates:
                    name = candidates[0]
                    extracted_path = temp_extract_dir / Path(name).name
                    # ensure file exists (it should, because we kept them)
                    if not extracted_path.exists():
                        with z.open(name) as src, open(extracted_path, "wb") as out:
                            out.write(src.read())
                    u = nv.show_file(str(extracted_path))
                    u.clear_representations()
                    u.add_cartoon(selection="chain A", color="blue")
                    u.add_cartoon(selection="chain B", color="red")
                    display(u)
        else:
            print("Could not find zip file for top model for visualization.")


Top model: fold_timp3_vs_adam17_c_aneglc_proteinmpnn_vadam17


NGLWidget()

In [10]:
# Cell 8: Guidance on interpreting outputs
from IPython.display import Markdown, display

text = """
# How to interpret the results and pick candidates for wet-lab testing

**Primary signals**
- **Interface area**: Larger interfaces (commonly > 600–800 Å²) are more likely to represent biologically meaningful, tight interfaces for protein–protein interactions. Small interfaces (< ~200 Å²) often indicate transient/weak contact or an artifact.
- **Number of contacts**: More inter-atomic contacts (< 5 Å) supports a plausible binding pose. Look for concentrated clusters of contacts (per-residue contact CSV / contact map) rather than sparse single-atom touches.
- **Clashes**: High number of severe clashes (atoms < ~2.2 Å) is a red flag. Good models should have near-zero severe clashes.
- **Centroid distance**: Very large centroid separations indicate non-interacting chains. Extremely small centroids with many clashes are also bad.
- **pLDDT**: AlphaFold's per-residue confidence (stored in B-factor) — mean pLDDT for chain B (the target) and for interface residues: values > 70–80 are more reliable. Prefer models where interface residues have decent pLDDT (> 70).
- **RMSD to reference** (if provided): Low RMSD to a known template/complex suggests a correct pose. Beware if reference has different numbering or missing residues.
- **PAE (Predicted Aligned Error)**: If PAE is available, check error between interface residues (low PAE between ligand and target interface residues supports confidence).

**Practical ranking strategy**
1. Filter out models with **many clashes** and **tiny interface area** (< ~200 Å²).
2. From remaining, prefer those with **higher Score** (the pipeline score combines contacts, interface area, clashes, centroid distance, and pLDDT bonus).
3. Inspect **top ~5–10** candidates manually:
   - Check per-residue pLDDT of interface residues (should be moderately high).
   - Inspect contact map and per-residue CSV; ensure the ligand touches a coherent patch, not a random chain end.
   - Use the interactive NGLview or PyMOL to look at geometry (hydrophobic pocket, salt-bridge network, H-bonds).
4. Optionally run more detailed tools (FoldX, Rosetta, PRODIGY) on the top picks.

**What to watch for in the export files**
- `binding_summary.csv` — quick ranking
- `*_per_residue_interface.csv` — which residues are near the partner, distances, pLDDT, and SASA
- `*_contact_map.csv` — distance matrix to visualize contact clusters
- `binding_dashboard.html` — interactive scatter with hover to compare many models quickly

If you share a few top model names, I can help inspect them manually and recommend which to test first.
"""

display(Markdown(text))



# How to interpret the results and pick candidates for wet-lab testing

**Primary signals**
- **Interface area**: Larger interfaces (commonly > 600–800 Å²) are more likely to represent biologically meaningful, tight interfaces for protein–protein interactions. Small interfaces (< ~200 Å²) often indicate transient/weak contact or an artifact.
- **Number of contacts**: More inter-atomic contacts (< 5 Å) supports a plausible binding pose. Look for concentrated clusters of contacts (per-residue contact CSV / contact map) rather than sparse single-atom touches.
- **Clashes**: High number of severe clashes (atoms < ~2.2 Å) is a red flag. Good models should have near-zero severe clashes.
- **Centroid distance**: Very large centroid separations indicate non-interacting chains. Extremely small centroids with many clashes are also bad.
- **pLDDT**: AlphaFold's per-residue confidence (stored in B-factor) — mean pLDDT for chain B (the target) and for interface residues: values > 70–80 are more reliable. Prefer models where interface residues have decent pLDDT (> 70).
- **RMSD to reference** (if provided): Low RMSD to a known template/complex suggests a correct pose. Beware if reference has different numbering or missing residues.
- **PAE (Predicted Aligned Error)**: If PAE is available, check error between interface residues (low PAE between ligand and target interface residues supports confidence).

**Practical ranking strategy**
1. Filter out models with **many clashes** and **tiny interface area** (< ~200 Å²).
2. From remaining, prefer those with **higher Score** (the pipeline score combines contacts, interface area, clashes, centroid distance, and pLDDT bonus).
3. Inspect **top ~5–10** candidates manually:
   - Check per-residue pLDDT of interface residues (should be moderately high).
   - Inspect contact map and per-residue CSV; ensure the ligand touches a coherent patch, not a random chain end.
   - Use the interactive NGLview or PyMOL to look at geometry (hydrophobic pocket, salt-bridge network, H-bonds).
4. Optionally run more detailed tools (FoldX, Rosetta, PRODIGY) on the top picks.

**What to watch for in the export files**
- `binding_summary.csv` — quick ranking
- `*_per_residue_interface.csv` — which residues are near the partner, distances, pLDDT, and SASA
- `*_contact_map.csv` — distance matrix to visualize contact clusters
- `binding_dashboard.html` — interactive scatter with hover to compare many models quickly

If you share a few top model names, I can help inspect them manually and recommend which to test first.


In [11]:
# Cell 9: Show top N candidates summary table
top_n = 10
if summary_df.shape[0] == 0:
    print("No models processed.")
else:
    display(summary_df.head(top_n))
    # Save a filtered CSV with top candidates
    summary_df.head(top_n).to_csv(output_folder / "top_candidates.csv", index=False)
    print(f"Top {min(top_n, summary_df.shape[0])} saved to top_candidates.csv")


,Model,Contacts,Unique_Pairs,Interface_Area,Centroid_Distance,Clashes,Mean_pLDDT_A,Mean_pLDDT_B,Mean_pLDDT,RMSD_to_ref_B,Score,PAE_present,FoldX,PRODIGY
93,fold_timp3_vs_adam17_c_aneglc_proteinmpnn_vadam17,114,114,0.000000e+00,26.480539,0,85.452923,92.152706,89.315861,None,84.961823,True,None,None
120,fold_timp3_vs_adam17_ef_tacd_proteinmpnn_vadam17,112,112,0.000000e+00,27.868567,0,85.544707,93.111111,89.907318,None,84.179946,True,None,None
125,fold_timp3_vs_adam17_ef_tscd_proteinmpnn_vmmp9,112,112,0.000000e+00,27.085501,0,85.519514,92.497897,89.543087,None,84.154443,True,None,None
19,fold_timp3_vs_adam10_ab_kgkyge_proteinmpnn_vad...,111,111,3.637979e-12,26.693525,0,87.107949,92.516514,90.124019,None,84.123305,True,None,None
117,fold_timp3_vs_adam17_ef_sncd_proteinmpnn_vmmp9,112,112,0.000000e+00,26.810165,0,85.075033,92.427391,89.314231,None,84.095082,True,None,None
119,fold_timp3_vs_adam17_ef_sscd_proteinmpnn_vmmp9,111,111,0.000000e+00,26.725742,0,85.820760,92.955145,89.934280,None,84.021991,True,None,None
124,fold_timp3_vs_adam17_ef_tscd_proteinmpnn_vadam17,111,111,0.000000e+00,27.320000,0,85.315584,92.398022,89.399152,None,83.635576,True,None,None
95,fold_timp3_vs_adam17_c_anenlc_proteinmpnn_vadam17,112,112,0.000000e+00,28.066290,0,84.565135,91.894011,88.790793,None,83.582139,True,None,None
112,fold_timp3_vs_adam17_ef_fscd_proteinmpnn_vadam17,110,110,0.000000e+00,27.318827,0,86.151245,92.917318,90.052404,None,83.562437,True,None,None
108,fold_timp3_vs_adam17_c_sveelc_proteinmpnn_vmmp9,112,112,0.000000e+00,27.392473,0,84.346878,91.200072,88.298269,None,83.470640,True,None,None


Top 10 saved to top_candidates.csv


In [12]:
# Cell 10: Final cleanup - delete extracted/temp files and optionally FoldX temporary dirs.
cleanup_confirm = True  # set False if you want to keep temp files for inspection

if cleanup_confirm:
    # Read list of extracted files if created
    extracted_list_path = output_folder / "extracted_files.txt"
    if extracted_list_path.exists():
        try:
            with open(extracted_list_path) as fh:
                lines = [l.strip() for l in fh if l.strip()]
            # remove those files
            for p in lines:
                try:
                    Path(p).unlink()
                except Exception:
                    pass
        except Exception:
            pass

    # Remove temp_extract_dir entirely
    try:
        shutil.rmtree(temp_extract_dir)
        print("Temporary extraction directory removed:", temp_extract_dir)
    except Exception as e:
        print("Failed to remove temp dir:", e)
else:
    print("Skipping cleanup; temp files kept in:", temp_extract_dir)


Temporary extraction directory removed: ..\Local\AlphaFoldMDA\af_extract_tmp
